# Harnesses: del modelo al agente

**Modelos Generativos Profundos · MIA · ICAI · 2026-2027**

En este tutorial usamos un modelo local (Qwen3.5-4B) como cerebro de tres harnesses:
**DeepSeek Harness**, **Codex** y **Claude Code**. Nos centramos en DeepSeek Harness.

Recorrido:

1. Una llamada al modelo (sin harness).
2. Un harness mínimo escrito a mano.
3. DeepSeek Harness: sus tools y una tool nuestra que genera documentos Word.
4. Codex y Claude Code sobre el mismo modelo (opcional): Codex analiza un Excel ejecutando
   Python y escribe un informe Word.

Todo el código está en `src/harness_lab/`. Aquí solo lo usamos.

## 0. Preparación

En una terminal, desde la carpeta del repositorio:

```bash
uv sync    # instala todo en .venv
```

Después abrid este notebook y elegid el kernel `.venv/bin/python`.

> Sin GPU podéis ensayar todo con un modelo simulado: `export HARNESS_SCRIPTED=1` antes de abrir
> el notebook. Las respuestas son fijas, pero el flujo es el mismo.

In [ ]:
import json, os, shutil
from pathlib import Path

import torch
from harness_lab import config

print("Modelo:      ", config.MODEL_ID)
print("Simulado:    ", config.SCRIPTED)
print("GPU:         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")
print("Workspace:   ", config.WORKSPACE_DIR)

## 1. El modelo

Qwen3.5-4B ocupa unos 9,3 GB en bf16. Cabe en la GPU de 16 GB de la DGX y deja sitio para
la caché KV. La primera descarga tarda unos minutos.

In [ ]:
from harness_lab.download import download

if not config.SCRIPTED:
    download(config.MODEL_ID)

Arrancamos el servidor local. Es un proceso aparte que carga el modelo una vez y ofrece
**tres APIs** distintas sobre el mismo modelo:

| Endpoint | API | Quién la usa |
|---|---|---|
| `/v1/chat/completions` | OpenAI Chat Completions | nuestras llamadas y el harness mínimo |
| `/v1/messages` | Anthropic Messages | DeepSeek Harness y Claude Code |
| `/v1/responses` | OpenAI Responses | Codex |

Código: `src/harness_lab/server/`. Cada petición se guarda en `outputs/traffic/`.

In [ ]:
from harness_lab.server.launch import start_server

server = start_server()   # unos 30-60 s la primera vez

## 2. Una llamada al modelo (sin harness)

Texto entra, texto sale. El modelo no recuerda nada entre llamadas y no ejecuta nada.

> **¿Por qué `from openai import OpenAI`?** Es solo una librería cliente HTTP para el formato
> *Chat Completions*, el estándar de facto (vLLM y SGLang sirven Qwen con ese mismo formato).
> Con `base_url=config.BASE_URL` habla con **nuestro servidor local**: el modelo que responde
> es Qwen y no se envía nada a OpenAI.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=config.BASE_URL, api_key=config.API_KEY)
reply = client.chat.completions.create(
    model=config.MODEL_ID,
    messages=[{"role": "user", "content": "Explica en una frase qué es un agente de IA."}],
)
print(reply.choices[0].message.content)

Ahora le damos una tool. El modelo **no la ejecuta**: solo pide que alguien la ejecute.
El servidor convierte el texto `<tool_call>...` del modelo en JSON (`server/toolcalls.py`).

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "create_word_document",
        "description": "Create a Word document from Markdown",
        "parameters": {
            "type": "object",
            "properties": {"filename": {"type": "string"}, "markdown": {"type": "string"}},
            "required": ["filename", "markdown"],
        },
    },
}]
reply = client.chat.completions.create(
    model=config.MODEL_ID,
    messages=[{"role": "user", "content": "Crea un documento Word con tres ideas sobre agentes."}],
    tools=tools,
)
msg = reply.choices[0].message
print("finish_reason:", reply.choices[0].finish_reason)
for call in msg.tool_calls or []:
    print(call.function.name, json.loads(call.function.arguments))

El modelo ha pedido la tool, pero no existe ningún fichero. Falta alguien que:
ejecute la tool, devuelva el resultado al modelo y repita hasta terminar.
**Eso es el harness.**

## 3. Un harness mínimo

`src/harness_lab/mini_harness.py` tiene unas 100 líneas. El bucle es:

1. Llamar al modelo con el historial y las tools.
2. Si no pide tools, terminar.
3. Si pide tools, ejecutarlas, añadir los resultados al historial y volver a 1.

In [ ]:
from harness_lab.mini_harness import MiniHarness

harness = MiniHarness()
print("Tools:", list(harness.registry))
answer = harness.run(
    "Crea un documento Word llamado agentes.docx con un título y tres ideas sobre agentes. Añade una tabla comparando un agente vs una llamada a openai. Dale un formato bonito al docx."
    "Después léelo para comprobarlo."
)
print("\nRespuesta:", answer)

In [ ]:
# El historial completo: esto es lo que el modelo ve en cada vuelta
for m in harness.messages:
    print(f"{m['role']:>9}: {str(m.get('content') or m.get('tool_calls'))[:150]}")

## 4. DeepSeek Harness

DeepSeek Harness (`dsh`, agosto 2026, MIT) es el harness abierto de DeepSeek. Idea central:
**todo es un plugin** (modelos, tools, skills, sesiones, sandbox, compactación).
Está escrito en TypeScript, pero `uv sync` ya instaló el SDK de Python con su runtime.

Su adaptador de DeepSeek habla la API Messages de Anthropic, así que lo apuntamos a nuestro
servidor con `DEEPSEEK_BASE_URL`. Código: `src/harness_lab/dsh.py`.

Perfiles:

| Perfil | Tools |
|---|---|
| `sdk-minimal` | `bash`, `read_image`, `working_directory` |
| `sdk` | ficheros, búsqueda, skills (Office), plan, todo, subagentes, web... |

Empezamos con `sdk-minimal` y sin tools nuestras.

In [ ]:
from harness_lab import dsh

result = dsh.run_task(
    "¿Qué ficheros hay en el directorio actual? Usa bash.",
    mcp=False,
    session_id="demo-bash",
)
print("\n", result.finish_reason, "|", result.final_response)

¿Qué le ha mandado el harness al modelo? El servidor guarda cada petición.
Mirad el *system prompt* y la lista de tools: eso lo pone el harness, no vosotros.

In [ ]:
from harness_lab import traffic

traffic.show(api="messages");

### Añadir una tool propia: documentos Word por MCP

MCP (*Model Context Protocol*) es el estándar para conectar tools a un harness.
Nuestro servidor MCP (`src/harness_lab/tools/mcp_server.py`) ofrece:

- `create_word_document(filename, markdown, title)`: Markdown a `.docx` con python-docx.
- `read_word_document(filename)` y `list_workspace()`.
- `compare_checkpoints(base, edited)`: compara dos checkpoints (no la usamos aquí).

`dsh.py` lo registra con un *patch* de configuración. Dentro del harness aparecen como
`mcp__lab__create_word_document`. Codex y Claude Code usan exactamente el mismo servidor.

In [ ]:
result = dsh.run_task(
    "Crea un documento Word llamado harness.docx que explique qué es un harness de agentes. "
    "Incluye un título, una introducción, una lista de componentes y una tabla con tres harnesses "
    "(DeepSeek Harness, Codex, Claude Code). Usa la herramienta create_word_document.",
    session_id="demo-word",
)
print("\n", result.finish_reason, "|", result.final_response)

El documento está en `workspace/harness.docx`. Descargadlo desde code-server y abridlo en Word.

### El perfil completo

Con `profile="sdk"` el harness añade muchas más tools. Comparad el tamaño del system prompt
y la lista de tools con el perfil mínimo. Más tools = más contexto = más trabajo para un
modelo pequeño.

In [ ]:
result = dsh.run_task("Di hola en una frase.", profile="sdk", mcp=False, session_id="demo-sdk")
s = traffic.show(api="messages", system_preview=300)

## 5. Codex y Claude Code (opcional)

Son programas aparte. En la DGX se instalan sin root:

```bash
bash scripts/install_clis.sh
```

Cada uno recibe su propia configuración dentro del repo (`.codex_home/`, `.claude_home/`)
apuntando al servidor local. Código: `src/harness_lab/clis.py`.

### Codex: de un Excel a un informe Word ejecutando Python

Codex trae de serie una **shell con sandbox** (`exec_command`): puede mirar ficheros, escribir un
script, ejecutarlo, leer el error y corregirlo. Para analizar datos y crear un Word no necesita
ninguna tool nuestra (por eso `mcp=False`): le basta con Python, pandas y python-docx.

El **sandbox** es parte del valor de Codex: los comandos solo pueden escribir en `workspace/`.
En Linux usa bubblewrap, que no funciona dentro de algunos contenedores (code-server en la DGX).
En ese caso hay que elegir: no ejecutar comandos, o ejecutarlos sin sandbox
(`HARNESS_CODEX_SANDBOX=off`), confiando en el aislamiento del contenedor.

Los datos son **ficticios**: el uso de una plataforma de LLMs en una empresa, una fila por día
laborable, departamento y modelo (enero-junio de 2026). Esconden algunas historias que un buen
análisis debería encontrar.

In [ ]:
from harness_lab.data import make_llm_usage

datos = make_llm_usage(config.WORKSPACE_DIR / "datos" / "uso_llm.xlsx")
print(datos.shape)
datos.head()

In [ ]:
from harness_lab import clis

TAREA = (
    "En datos/uso_llm.xlsx hay datos ficticios del uso de una plataforma de LLMs en una empresa "
    "(una fila por día, departamento y modelo). Analízalos escribiendo y ejecutando un script de "
    "Python con pandas: no inventes ninguna cifra. Después genera con python-docx el informe "
    "informe_uso_llm.docx con: 1) un resumen con el total de peticiones y el coste total en euros, "
    "2) una tabla de coste por departamento, 3) una tabla de peticiones y coste por modelo, "
    "4) el coste por mes y 5) tres conclusiones basadas en las cifras."
)
# Codex ejecuta cada comando en su sandbox (bubblewrap). Dentro de algunos contenedores, como
# code-server en la DGX, ese sandbox no puede arrancar. Aquí el contenedor ya aísla, así que
# Codex ejecuta los comandos SIN sandbox: arranca en workspace/, pero nada le impide escribir
# fuera. Borrad esta línea para volver al sandbox en un entorno donde funcione.
os.environ["HARNESS_CODEX_SANDBOX"] = "off"

skip = os.environ.get("HARNESS_SKIP_CLIS") == "1"
if shutil.which("codex") and not skip:
    print(clis.run_codex(TAREA, mcp=False))
else:
    print("codex no está instalado" if not shutil.which("codex") else "omitido (HARNESS_SKIP_CLIS=1)")

¿Qué ha hecho Codex? Estos son los comandos que pidió ejecutar (salen del tráfico guardado):

In [ ]:
from harness_lab import traffic

if shutil.which("codex") and not skip:
    traffic.show_tool_calls(api="responses")

El informe, y la comprobación: **no os fiéis de un agente sin verificar**. Comparad sus cifras
con el cálculo directo.

In [ ]:
from harness_lab.tools.docx_tools import read_word_document

informe = config.WORKSPACE_DIR / "informe_uso_llm.docx"
if informe.exists():
    print(read_word_document(informe.name))
else:
    print("No hay informe (Codex no se ha ejecutado o no lo ha generado).")

print("\n--- Cálculo directo con pandas ---")
print(f"Peticiones: {datos.peticiones.sum():,}   Coste total: {datos.coste_eur.sum():,.2f} €")
print(datos.groupby("departamento").coste_eur.sum().round(2).sort_values(ascending=False).to_string())

### Claude Code (No hice muchas pruebas con este)

Mismo modelo, otro harness. Aquí con nuestra tool MCP de Word.

In [ ]:
from harness_lab import clis, traffic

skip = os.environ.get("HARNESS_SKIP_CLIS") == "1"
if shutil.which("claude") and not skip:
    print(clis.run_claude("Crea un documento Word claude.docx con dos frases sobre Claude Code."))
    traffic.show(api="messages", system_preview=300)
else:
    print("claude no está instalado" if not shutil.which("claude") else "omitido (HARNESS_SKIP_CLIS=1)")

Mismo modelo, tres harnesses. Cambian el system prompt, las tools, los permisos y el bucle.
Por eso el mismo modelo se comporta distinto en cada uno.